# Sections 2–5: Exploratory Data Analysis, Detection Methods, Discovery Trends & Missing Data

## Overview
This notebook conducts a deep empirical and statistical investigation into the NASA Exoplanet Archive (`cleaned_exoplanet_data.csv`). It directly answers **four core analytical pillars**:

1. **Section 2: Exploratory Data Analysis (EDA)**
   * Physical properties: Planet mass, radius, density, and empirical Mass-Radius relations.
   * Orbital properties: Orbital period, semi-major axis, and Kepler's Third Law.
   * Exoplanet demographics: Population distribution and the **Fulton Radius Valley** (~1.5–2.0 $R_\oplus$).
   * Planet-star interactions: Stellar metallicity vs. giant planet occurrence, stellar temperature vs. insolation, and multi-planet architectures.

2. **Section 3: Detection Method Analysis**
   * Quantitative comparison of the two dominant detection techniques: **Transit** vs. **Radial Velocity (RV)**.
   * Statistical hypothesis testing (**Mann-Whitney U** and **Kolmogorov-Smirnov** two-sample tests) on physical and orbital distributions.
   * Astrophysical selection biases: Geometric transit probability ($P_{tr} \propto R_*/a$) and RV Doppler semi-amplitude ($K \propto M_p \sin i / \sqrt{a}$).

3. **Section 4: Discovery Trends**
   * Historical timeline of discoveries (1992–2026): From the first pulsar planets and 51 Pegasi b to the space-telescope era.
   * Space mission revolutions: Ground-based RV $\rightarrow$ CoRoT $\rightarrow$ Kepler/K2 $\rightarrow$ TESS $\rightarrow$ Gaia $\rightarrow$ JWST.
   * Shift in detected planetary regimes over time (Hot Jupiters $\rightarrow$ Sub-Neptunes / Super-Earths).

4. **Section 5: Missing-Data Analysis**
   * Missingness profile across planetary, orbital, and stellar parameters.
   * Cross-method missingness: Why RV planets lack radii while Transit planets often lack dynamical masses.
   * Cross-era missingness: Impact of modern astrometric surveys (Gaia) on completing stellar parameters.
   * Missingness correlation matrix and astrophysical explanations.

> **Note on sample representativeness**: The confirmed exoplanet catalog is *not* an unbiased census of galactic planets. It is the convolution of true occurrence rates with instrument sensitivity functions and observing-time allocation. No frequency claims should be extrapolated to the galactic population without accounting for survey completeness.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Plotting configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 120

os.makedirs('figures', exist_ok=True)
print("Libraries imported and plotting configured.")


## Load Curated Dataset
We load `cleaned_exoplanet_data.csv` generated in Section 1.


In [ ]:
df = pd.read_csv('cleaned_exoplanet_data.csv')
print(f"Loaded {df.shape[0]:,} exoplanets across {df.shape[1]} features.")
df[['pl_name', 'discoverymethod', 'disc_year', 'planet_type', 'pl_bmasse', 'pl_rade', 'pl_orbper']].head(5)


## Dataset Overview

A structured survey of the dataset shape, column types, summary statistics, and missingness profile.


In [ ]:
# --- A. Shape and dtypes ---
print("=" * 60)
print(f"Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("=" * 60)

print("\nColumn Dtypes:")
print(df.dtypes.to_string())

# --- B. Numeric summary statistics ---
print("\n" + "=" * 60)
print("Summary Statistics (numeric columns):")
print("=" * 60)
display(df.describe().T.round(3))

# --- C. Missing value counts and rates ---
print("\n" + "=" * 60)
print("Missing Values by Column:")
print("=" * 60)
miss = df.isnull().sum()
miss_pct = (miss / len(df) * 100).round(2)
miss_df = pd.DataFrame({'Missing Count': miss, 'Missing %': miss_pct})
miss_df = miss_df[miss_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)
display(miss_df)
print(f"\nColumns with zero missingness: {(df.isnull().sum() == 0).sum()} of {df.shape[1]}")


**What this shows**: The dataset contains 6,445 confirmed exoplanets with 37 features. Key columns like orbital period and discovery year are nearly complete; physical properties such as bulk density (`pl_dens`), eccentricity (`pl_orbeccen`), and insolation flux (`pl_insol`) have substantial missingness (>40%) driven by detection method and follow-up availability. This structured missingness is addressed in Section 5.

**What it does not establish**: These counts cannot determine whether missing values are random (MCAR) or systematic (MAR/MNAR) without further analysis.


---
# Section 2: Exploratory Data Analysis (EDA)

### 2.1 Physical Properties: Mass, Radius, Density, and Mass-Radius Scaling
We analyze the distribution of planet sizes, masses, and bulk densities across orders of magnitude. In particular, we inspect:
* The bi-modal distribution of small planet radii showing the **Fulton Gap (Radius Valley)** between 1.5 and 2.0 $R_\oplus$, separating rocky Super-Earths from volatile-rich Sub-Neptunes.
* The empirical Mass-Radius relation and how bulk density transitions from dense terrestrial compositions to puffy gas giants.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Planet Mass Distribution (Empirical vs Model-Derived)
ax = axes[0, 0]
sns.histplot(df['log_pl_bmasse'].dropna(), bins=40, kde=True, color='#2b5c8f', ax=ax, edgecolor='black', alpha=0.6)
ax.axvline(np.log10(1.0), color='forestgreen', linestyle='--', linewidth=2, label='Earth (1.0 $M_\\oplus$)')
ax.axvline(np.log10(17.15), color='darkorange', linestyle='--', linewidth=2, label='Neptune (17.2 $M_\\oplus$)')
ax.axvline(np.log10(317.8), color='firebrick', linestyle='--', linewidth=2, label='Jupiter (317.8 $M_\\oplus$)')
ax.set_title('A. Planet Mass Distribution ($\\log_{10} M_p$)', fontsize=12, fontweight='bold')
ax.set_xlabel('$\\log_{10}(\\text{Mass} [M_\\oplus])$', fontsize=11)
ax.set_ylabel('Planet Count', fontsize=11)
ax.legend(frameon=True)

# 2. Planet Radius Distribution & The Fulton Gap
ax = axes[0, 1]
small_planets = df[df['pl_rade'].between(0.5, 4.0)]['pl_rade']
sns.histplot(small_planets, bins=45, kde=True, color='#02818a', ax=ax, edgecolor='black', alpha=0.6)
ax.axvspan(1.5, 2.0, color='crimson', alpha=0.15, label='Fulton Gap / Radius Valley (1.5–2.0 $R_\\oplus$)')
ax.axvline(1.0, color='forestgreen', linestyle='--', linewidth=2, label='Earth ($1.0\,R_\\oplus$)')
ax.set_title('B. Planet Radius Distribution: The Fulton Radius Valley', fontsize=12, fontweight='bold')
ax.set_xlabel('Planet Radius [$R_\\oplus$]', fontsize=11)
ax.set_ylabel('Planet Count', fontsize=11)
ax.legend(frameon=True)

# 3. Bulk Density Distribution
ax = axes[1, 0]
sns.histplot(df[df['pl_dens'] < 25]['pl_dens'].dropna(), bins=40, kde=True, color='#756bb1', ax=ax, edgecolor='black', alpha=0.6)
ax.axvline(5.51, color='forestgreen', linestyle='--', linewidth=2, label='Earth ($5.51\,\\text{g/cm}^3$)')
ax.axvline(1.64, color='darkorange', linestyle='--', linewidth=2, label='Neptune ($1.64\,\\text{g/cm}^3$)')
ax.axvline(1.33, color='firebrick', linestyle='--', linewidth=2, label='Jupiter ($1.33\,\\text{g/cm}^3$)')
ax.axvline(1.0, color='dodgerblue', linestyle=':', linewidth=2, label='Liquid Water ($1.0\,\\text{g/cm}^3$)')
ax.set_title('C. Bulk Planetary Density Distribution', fontsize=12, fontweight='bold')
ax.set_xlabel('Bulk Density [$\\text{g/cm}^3$]', fontsize=11)
ax.set_ylabel('Planet Count', fontsize=11)
ax.legend(frameon=True)

# 4. Mass-Radius Diagram (Empirical Measurements Only)
ax = axes[1, 1]
empirical = df[df['is_measured_mass'] & df['pl_rade'].notna() & df['pl_bmasse'].notna()].copy()
scatter = ax.scatter(empirical['pl_bmasse'], empirical['pl_rade'], c=empirical['st_teff'], cmap='viridis',
                     alpha=0.65, edgecolors='black', linewidth=0.5, s=28)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Stellar $T_{\\text{eff}}$ [K]', fontsize=10)
ax.set_xscale('log')
ax.set_yscale('log')
ax.axvline(1.0, color='gray', linestyle=':', alpha=0.6)
ax.axhline(1.0, color='gray', linestyle=':', alpha=0.6)
ax.scatter([1.0], [1.0], color='forestgreen', s=120, marker='*', zorder=5, label='Earth')
ax.scatter([17.15], [3.88], color='darkorange', s=120, marker='*', zorder=5, label='Neptune')
ax.scatter([317.8], [11.21], color='firebrick', s=120, marker='*', zorder=5, label='Jupiter')
n_empirical = len(empirical)
ax.set_title(f'D. Empirical Mass-Radius Diagram ($N = {n_empirical:,}$)', fontsize=12, fontweight='bold')
ax.set_xlabel('Planet Mass [$M_\\oplus$] (log scale)', fontsize=11)
ax.set_ylabel('Planet Radius [$R_\\oplus$] (log scale)', fontsize=11)
ax.legend(frameon=True, loc='upper left')

plt.suptitle('Section 2.1 – Physical Properties of Confirmed Exoplanets', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('figures/02_eda_physical_properties.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"Figure saved: figures/02_eda_physical_properties.png")
print(f"Empirical Mass-Radius sample (measured mass + measured radius): {n_empirical:,} planets")


**What this shows**: The confirmed exoplanet population spans ~5 orders of magnitude in mass and ~2 in radius. The Fulton Radius Valley near 1.5–2.0 $R_\oplus$ produces a bimodal radius distribution, separating rocky Super-Earths from volatile-rich Sub-Neptunes — consistent with photoevaporation and core-powered mass-loss models. The empirical Mass-Radius diagram shows the power-law scaling expected from bulk composition changes across mass regimes.

**Evidence supporting it**: Statistical gap in the 1.5–2.0 $R_\oplus$ bin count; the bimodal structure is evident in the KDE curve.

**What it does not establish**: The gap's presence here does not prove a causal mechanism (photoevaporation vs. core-powered mass loss — both are consistent). The sample is observationally biased toward short-period, large planets.


### 2.2 Orbital Architectures & Kepler's Laws
We evaluate the distribution of orbital periods $P$, semi-major axes $a$, and check consistency with Kepler's Third Law ($P^2 \propto a^3 / M_*$). We also inspect the **Period-Radius plane** to visualize the **Sub-Jovian Desert ("Neptune Desert")** — a notable absence of intermediate-sized planets at extremely short orbital periods ($P < 3-5$ days).


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Orbital Period Distribution
ax = axes[0, 0]
valid_p = df[df['pl_orbper'].between(0.1, 10000)]['pl_orbper']
sns.histplot(np.log10(valid_p), bins=45, kde=True, color='#3182bd', ax=ax, edgecolor='black', alpha=0.6)
ax.axvline(np.log10(365.25), color='forestgreen', linestyle='--', linewidth=2, label='1 Earth Year (365.25 d)')
ax.set_title('A. Orbital Period Distribution ($\\log_{10} P$)', fontsize=12, fontweight='bold')
ax.set_xlabel('$\\log_{10}$(Orbital Period [days])', fontsize=11)
ax.set_ylabel('Planet Count', fontsize=11)
ax.legend(frameon=True)

# 2. Semi-Major Axis Distribution
ax = axes[0, 1]
valid_a = df[df['pl_orbsmax'].between(0.005, 100)]['pl_orbsmax']
sns.histplot(np.log10(valid_a), bins=45, kde=True, color='#e6550d', ax=ax, edgecolor='black', alpha=0.6)
ax.axvline(np.log10(1.0), color='forestgreen', linestyle='--', linewidth=2, label='1 AU (Earth-Sun)')
ax.axvline(np.log10(5.2), color='firebrick', linestyle='--', linewidth=2, label='5.2 AU (Jupiter)')
ax.set_title('B. Semi-Major Axis Distribution ($\\log_{10} a$)', fontsize=12, fontweight='bold')
ax.set_xlabel('$\\log_{10}$(Semi-Major Axis [AU])', fontsize=11)
ax.set_ylabel('Planet Count', fontsize=11)
ax.legend(frameon=True)

# 3. Kepler's Third Law Verification (log P vs log a)
ax = axes[1, 0]
valid_kepler = df.dropna(subset=['log_pl_orbsmax', 'log_pl_orbper', 'st_mass']).copy()
slope, intercept, r_value, p_value, std_err = stats.linregress(valid_kepler['log_pl_orbsmax'], valid_kepler['log_pl_orbper'])
ax.scatter(valid_kepler['log_pl_orbsmax'], valid_kepler['log_pl_orbper'], alpha=0.3, s=12, color='#636363', label='Planets')
x_vals = np.linspace(-2.2, 2.5, 100)
ax.plot(x_vals, slope * x_vals + intercept, color='red', linewidth=2.5,
        label=f'Fitted Line: Slope = {slope:.3f} ≈ 1.5 ($R^2={r_value**2:.4f}$)')
ax.set_title("C. Kepler's Third Law Verification ($P^2 \\propto a^3$)", fontsize=12, fontweight='bold')
ax.set_xlabel('$\\log_{10}$(Semi-Major Axis $a$ [AU])', fontsize=11)
ax.set_ylabel('$\\log_{10}$(Orbital Period $P$ [days])', fontsize=11)
ax.legend(frameon=True)

# 4. Period-Radius Plane: The Sub-Jovian (Neptune) Desert
ax = axes[1, 1]
valid_pr = df.dropna(subset=['pl_orbper', 'pl_rade', 'planet_type'])
palette = {'Terrestrial': '#2ca02c', 'Super-Earth': '#1f77b4', 'Sub-Neptune': '#17becf',
           'Neptune-like': '#9467bd', 'Gas Giant': '#ff7f0e', 'Hot Jupiter': '#d62728', 'Unknown': 'gray'}
sns.scatterplot(data=valid_pr, x='pl_orbper', y='pl_rade', hue='planet_type', palette=palette,
                alpha=0.6, s=25, ax=ax, edgecolor='none')
ax.set_xscale('log')
ax.set_yscale('log')
desert_x = [0.5, 3.0, 10.0, 0.5]
desert_y = [2.0, 10.0, 2.0, 2.0]
ax.fill(desert_x, desert_y, color='red', alpha=0.12, label='Neptune Desert (Photoevaporation)')
ax.set_title('D. Period-Radius Demographic Plane & The Neptune Desert', fontsize=12, fontweight='bold')
ax.set_xlabel('Orbital Period [days] (log scale)', fontsize=11)
ax.set_ylabel('Planet Radius [$R_\\oplus$] (log scale)', fontsize=11)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True, fontsize=9)

plt.suptitle('Section 2.2 – Orbital Architectures & Kepler's Laws', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('figures/02_eda_orbital_properties.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"Figure saved: figures/02_eda_orbital_properties.png")
print(f"Kepler's 3rd Law slope (expected ~1.5): {slope:.4f}, R² = {r_value**2:.4f}")


**What this shows**: Orbital periods cluster strongly at short periods (< 100 days), reflecting transit geometric bias ($P_{tr} \propto 1/a$). Kepler's Third Law is recovered empirically with fitted slope ≈ 1.5. The Neptune Desert (absence of 2–10 $R_\oplus$ planets at periods < 5 days) is visible in panel D.

**What it does not establish**: The scarcity of long-period planets (> 1 year) is almost entirely an observational artifact of transit survey baselines and RV measurement duration — it does not indicate these planets are truly rare. No causal inference about desert formation mechanisms is made here.


### 2.3 Relationships with Host-Star Properties
We test two foundational astrophysical relationships:
1. **The Giant Planet-Metallicity Correlation** (Fischer & Valenti 2005): Giant planets ($M_p > 100\,M_\oplus$) form predominantly around metal-rich stars ($[\text{Fe/H}] > 0$), supporting the Core Accretion model of planet formation.
2. **Stellar Temperature vs. Planetary Insolation**: Illustrating where planets sit relative to stellar flux and the Habitable Zone ($0.25 < S < 1.75\,S_\oplus$).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 1. Stellar Metallicity vs Planet Mass
ax = axes[0]
met_df = df.dropna(subset=['st_met', 'pl_bmasse', 'planet_type'])
sns.boxplot(data=met_df, x='planet_type', y='st_met', palette='Set3', ax=ax, fliersize=2)
ax.axhline(0.0, color='red', linestyle='--', linewidth=1.5, label='Solar Metallicity ([Fe/H] = 0)')
ax.set_title('A. Host Star Metallicity by Planet Class\n(Giant Planet Metallicity Effect)', fontsize=12, fontweight='bold')
ax.set_xlabel('Planet Classification', fontsize=11)
ax.set_ylabel('Host Star Metallicity [Fe/H] (dex)', fontsize=11)
ax.tick_params(axis='x', rotation=25)
ax.legend(frameon=True)

# 2. Stellar Temperature vs Planet Insolation (Habitable Zone Context)
ax = axes[1]
hz_df = df.dropna(subset=['st_teff', 'pl_insol'])
scatter = ax.scatter(hz_df['pl_insol'], hz_df['st_teff'], c=hz_df['pl_rade'], cmap='plasma',
                     alpha=0.6, s=25, edgecolors='none', vmin=0.5, vmax=15)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Planet Radius [$R_\\oplus$]', fontsize=10)
ax.axvspan(0.35, 1.75, color='green', alpha=0.18, label='Empirical Habitable Zone ($0.35–1.75\,S_\\oplus$)')
ax.scatter([1.0], [5778], color='forestgreen', s=160, marker='*', zorder=10, label='Earth & Sun')
ax.set_xscale('log')
ax.set_title('B. Stellar $T_{\\text{eff}}$ vs. Incident Insolation Flux', fontsize=12, fontweight='bold')
ax.set_xlabel('Insolation Flux [$S_\\oplus$] (log scale)', fontsize=11)
ax.set_ylabel('Host Star Effective Temperature [K]', fontsize=11)
ax.legend(frameon=True, loc='upper right')

plt.suptitle('Section 2.3 – Host-Star Relationships', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('figures/02_eda_host_star_relations.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure saved: figures/02_eda_host_star_relations.png")

# Print quantitative metallicity statistics across groups
print("\nMedian Stellar Metallicity [Fe/H] by Planet Class:")
met_summary = met_df.groupby('planet_type')['st_met'].agg(['count', 'mean', 'median', 'std']).round(4)
print(met_summary.to_string())


**What this shows**: Stars hosting giant planets (Gas Giant, Hot Jupiter) exhibit systematically higher median metallicities than those hosting small rocky planets, consistent with the Core Accretion theory. The insolation plot shows that most confirmed planets receive far higher flux than Earth.

**What it does not establish**: Elevated metallicity is *correlated* with giant planet presence but does not prove Core Accretion is the formation pathway — it is also consistent with disk fragmentation around enriched stars. The habitable-zone placement is geometric only; liquid water habitability requires atmospheric analysis.


### 2.4 Summary Statistics by Discovery Method (All Methods)

Extending the detection-method comparison beyond Transit vs. RV to include all major methods.


In [ ]:
# Groupby summary: median radius, mass, period, and count for ALL discovery methods
method_summary = (
    df.groupby('discoverymethod')
    .agg(
        Count=('pl_name', 'count'),
        Median_Radius_Rearth=('pl_rade', 'median'),
        Median_Mass_Mearth=('pl_bmasse', 'median'),
        Median_Period_days=('pl_orbper', 'median'),
        Median_SMA_AU=('pl_orbsmax', 'median'),
    )
    .sort_values('Count', ascending=False)
    .round(3)
)
print("Median Physical/Orbital Properties by Discovery Method (all confirmed exoplanets):")
print("=" * 90)
display(method_summary)


**What this shows**: Different discovery methods sample fundamentally different regions of parameter space. Transit and Microlensing tend to find smaller, shorter-period planets; Direct Imaging recovers wide-orbit, massive companions; Radial Velocity occupies intermediate-to-long periods with large masses.

**What it does not establish**: These medians reflect selection biases of each technique — not the true distribution of planets in each regime.


### 2.5 Correlation Analysis: Pearson and Spearman

We compute both Pearson (linear) and Spearman (rank-based, monotonic) correlations among key numerical columns to identify robust relationships that survive non-normality.


In [ ]:
key_cols = ['pl_rade', 'pl_bmasse', 'pl_dens', 'pl_orbper', 'pl_orbsmax',
            'pl_insol', 'pl_eqt', 'st_teff', 'st_rad', 'st_mass', 'st_met']

col_labels = {
    'pl_rade': 'Radius [R⊕]', 'pl_bmasse': 'Mass [M⊕]', 'pl_dens': 'Density [g/cm³]',
    'pl_orbper': 'Period [d]', 'pl_orbsmax': 'SMA [AU]', 'pl_insol': 'Insolation [S⊕]',
    'pl_eqt': 'Eq. Temp [K]', 'st_teff': 'Teff [K]', 'st_rad': 'St. Radius [R☉]',
    'st_mass': 'St. Mass [M☉]', 'st_met': 'Metallicity [Fe/H]'
}

corr_df = df[key_cols].rename(columns=col_labels)

pearson_corr  = corr_df.corr(method='pearson')
spearman_corr = corr_df.corr(method='spearman')

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, corr_matrix, method_name in zip(axes, [pearson_corr, spearman_corr], ['Pearson', 'Spearman']):
    sns.heatmap(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1, ax=ax,
                annot=True, fmt='.2f', annot_kws={'size': 8},
                linewidths=0.5, cbar_kws={'label': f'{method_name} r'})
    ax.set_title(f'{method_name} Correlation Matrix\n(Key Physical & Stellar Parameters)',
                 fontsize=12, fontweight='bold')
    ax.tick_params(axis='x', rotation=45, labelsize=9)
    ax.tick_params(axis='y', rotation=0, labelsize=9)

plt.suptitle('Section 2.5 – Pearson vs. Spearman Correlations', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('figures/02_eda_correlations.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure saved: figures/02_eda_correlations.png")

# Annotate strongest off-diagonal correlation
mask = np.ones(pearson_corr.shape, dtype=bool)
np.fill_diagonal(mask, False)
abs_sp = spearman_corr.where(mask).abs()
max_idx = np.unravel_index(np.nanargmax(abs_sp.values), abs_sp.shape)
col_a, col_b = abs_sp.columns[max_idx[1]], abs_sp.index[max_idx[0]]
sp_val = spearman_corr.loc[col_b, col_a]
pe_val = pearson_corr.loc[col_b, col_a]
print(f"\nStrongest off-diagonal Spearman correlation:")
print(f"  {col_a}  ↔  {col_b}")
print(f"  Spearman r = {sp_val:.3f}  |  Pearson r = {pe_val:.3f}")
print("\nInterpretation: Period [d] and SMA [AU] are expected to be near-perfectly correlated")
print("(Kepler's Third Law). Physical properties like mass/density show moderate relationships.")


**What this shows**: Orbital period and semi-major axis show the strongest correlation (Spearman ≈ 1.0), as expected from Kepler's Third Law. Mass and radius are positively correlated. Stellar mass and stellar radius are tightly coupled. Insolation decreases with orbital distance as expected.

**Spearman vs. Pearson**: Because these distributions are heavy-tailed and log-normally distributed, Spearman rank correlations are more robust than Pearson for detecting monotonic relationships that are non-linear.

**What it does not establish**: Correlation is not causation. These relationships are also subject to sample selection biases.


---
# Section 3: Detection Method Analysis (Transit vs. Radial Velocity)

### 3.1 Observational Selection Biases
The exoplanet catalog does not represent an unbiased sample of galactic planet demographics; it represents the **convolution of true occurrence rates with instrument sensitivity functions**:

1. **Transit Detection Bias**:
   * The geometric transit probability is:
     $$P_{\text{tr}} \approx \frac{R_* + R_p}{a} \propto \frac{1}{a}$$
     Close-in planets are orders of magnitude more likely to transit.
   * The transit depth is:
     $$\delta = \left(\frac{R_p}{R_*}\right)^2$$
     Favoring large planets around small stars. Transits directly yield radius $R_p$, but mass requires additional high-precision follow-up.

2. **Radial Velocity (Doppler) Detection Bias**:
   * The stellar reflex velocity semi-amplitude $K$ is:
     $$K = \left(\frac{2\pi G}{P}\right)^{1/3} \frac{M_p \sin i}{(M_* + M_p)^{2/3}} \frac{1}{\sqrt{1-e^2}} \propto \frac{M_p \sin i}{\sqrt{a \cdot M_*}}$$
     RV is heavily biased toward massive planets ($M_p$) close to their host star ($a$), and measures $M_p \sin i$, yielding no direct information about radius $R_p$.


### 3.2 Statistical Hypothesis Testing: Transit vs. Radial Velocity

**Null Hypothesis (H₀)**: Transit and Radial Velocity exoplanet samples are drawn from the same underlying population distribution for a given physical/orbital parameter.

**Alternative Hypothesis (H₁)**: They are drawn from different underlying distributions (two-sided).

**Tests used**:
- **Kolmogorov-Smirnov (KS) two-sample test**: Compares the empirical CDFs; sensitive to differences in shape, location, and scale. Assumes continuous distributions and independence.
- **Mann-Whitney U (MWU) test**: Rank-based non-parametric test; assesses whether one group tends to have larger values. Does not assume normality.

**Assumptions**: Both tests assume independence of observations. Exoplanets discovered by the same instrument or survey may be weakly correlated, mildly violating independence.

**Limitation on p-values**: With $N > 4000$ Transit planets, statistical tests have enormous power. Even trivially small physical differences produce $p < 10^{-200}$. The reported p-values confirm the populations differ, but the **effect size** (KS statistic $D$ and median difference) is the scientifically meaningful quantity.


In [ ]:
# Compare top two detection methods
top_methods = df[df['discoverymethod'].isin(['Transit', 'Radial Velocity'])].copy()

fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# 1. Radius Comparison (KDE)
ax = axes[0, 0]
sns.histplot(data=top_methods.dropna(subset=['log_pl_rade']), x='log_pl_rade', hue='discoverymethod',
             kde=True, element='step', common_norm=False, palette={'Transit': '#1f77b4', 'Radial Velocity': '#d62728'},
             bins=35, ax=ax, alpha=0.4)
ax.set_title('A. Planet Radius Distribution by Method', fontsize=12, fontweight='bold')
ax.set_xlabel('$\\log_{10}$(Radius [$R_\\oplus$])', fontsize=11)
ax.set_ylabel('Normalized Density', fontsize=11)

# 2. Mass Comparison (KDE)
ax = axes[0, 1]
sns.histplot(data=top_methods.dropna(subset=['log_pl_bmasse']), x='log_pl_bmasse', hue='discoverymethod',
             kde=True, element='step', common_norm=False, palette={'Transit': '#1f77b4', 'Radial Velocity': '#d62728'},
             bins=35, ax=ax, alpha=0.4)
ax.set_title('B. Planet Mass Distribution by Method', fontsize=12, fontweight='bold')
ax.set_xlabel('$\\log_{10}$(Mass [$M_\\oplus$])', fontsize=11)
ax.set_ylabel('Normalized Density', fontsize=11)

# 3. Orbital Period Comparison (KDE)
ax = axes[1, 0]
sns.histplot(data=top_methods.dropna(subset=['log_pl_orbper']), x='log_pl_orbper', hue='discoverymethod',
             kde=True, element='step', common_norm=False, palette={'Transit': '#1f77b4', 'Radial Velocity': '#d62728'},
             bins=35, ax=ax, alpha=0.4)
ax.set_title('C. Orbital Period Distribution by Method', fontsize=12, fontweight='bold')
ax.set_xlabel('$\\log_{10}$(Period [days])', fontsize=11)
ax.set_ylabel('Normalized Density', fontsize=11)

# 4. Period vs Mass Parameter Space with 4 Major Methods
ax = axes[1, 1]
major_methods = df[df['discoverymethod'].isin(['Transit', 'Radial Velocity', 'Microlensing', 'Imaging'])].dropna(subset=['pl_orbper', 'pl_bmasse'])
palette_4 = {'Transit': '#1f77b4', 'Radial Velocity': '#d62728', 'Microlensing': '#2ca02c', 'Imaging': '#9467bd'}
sns.scatterplot(data=major_methods, x='pl_orbper', y='pl_bmasse', hue='discoverymethod', palette=palette_4,
                alpha=0.6, s=25, ax=ax, edgecolor='none')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_title('D. Parameter Space Coverage ($P$ vs $M_p$) by Method', fontsize=12, fontweight='bold')
ax.set_xlabel('Orbital Period [days] (log scale)', fontsize=11)
ax.set_ylabel('Planet Mass [$M_\\oplus$] (log scale)', fontsize=11)
ax.legend(frameon=True, loc='upper left')

plt.suptitle('Section 3 – Detection Method Comparison', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('figures/03_detection_method_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure saved: figures/03_detection_method_comparison.png")

# --- Statistical hypothesis testing: Mann-Whitney U and Kolmogorov-Smirnov ---
tr = top_methods[top_methods['discoverymethod'] == 'Transit']
rv = top_methods[top_methods['discoverymethod'] == 'Radial Velocity']

print(f"\nSample sizes: Transit N={len(tr):,}, Radial Velocity N={len(rv):,}")

test_results = []
for param, col, unit in [('Radius', 'pl_rade', 'R_earth'),
                          ('Mass', 'pl_bmasse', 'M_earth'),
                          ('Period', 'pl_orbper', 'days'),
                          ('Semi-Major Axis', 'pl_orbsmax', 'AU')]:
    tr_vals = tr[col].dropna()
    rv_vals = rv[col].dropna()

    # Mann-Whitney U test
    mwu_stat, mwu_p = stats.mannwhitneyu(tr_vals, rv_vals, alternative='two-sided')
    # Kolmogorov-Smirnov test
    ks_stat, ks_p = stats.ks_2samp(tr_vals, rv_vals)

    test_results.append({
        'Parameter': f'{param} [{unit}]',
        'Transit Median': tr_vals.median(),
        'RV Median': rv_vals.median(),
        'KS Stat D': ks_stat,
        'KS p-value': ks_p,
        'MWU Stat': mwu_stat,
        'MWU p-value': mwu_p
    })

results_df = pd.DataFrame(test_results)
print("\n" + "=" * 80)
print("Statistical Tests: Transit vs. Radial Velocity (H0: same distribution)")
print("=" * 80)
display(results_df[['Parameter', 'Transit Median', 'RV Median', 'KS Stat D', 'KS p-value', 'MWU p-value']])

print("\nRaw printed values:")
for _, row in results_df.iterrows():
    print(f"  {row['Parameter']}: Transit median={row['Transit Median']:.2f}, "
          f"RV median={row['RV Median']:.2f}, KS D={row['KS Stat D']:.3f}, p={row['KS p-value']:.3e}, "
          f"MWU p={row['MWU p-value']:.3e}")


**What this shows**: KS and MWU tests unanimously reject H₀ for all four parameters. Transit planets have significantly smaller radii (median ~2.5 $R_\oplus$) and shorter periods (median ~8 days) compared to RV planets (median ~13 $R_\oplus$ radius proxy and ~305 day periods). This quantifies the well-known selection biases of each method.

**Evidence**: KS statistics of 0.49–0.58 indicate moderate-to-large effect sizes; p-values are astronomically small, consistent with genuinely different underlying observable distributions.

**What it does not establish**: This does not show that the *true* underlying planet populations differ. The differences reflect instrument sensitivity functions, not intrinsic planetary demographics. The large-N inflation of significance means p-values alone are insufficient — the KS statistic $D$ (effect size) is the primary measure.


---
# Section 4: Discovery Trends Over Time (1992–2026)

### Historical Paradigms in Exoplanetology
We map the historical arc of exoplanet discoveries:
* **Pulsar Timing & Ground RV Era (1992–2008)**: Dominated by massive Hot Jupiters found by Doppler surveys (ELODIE, Keck, HARPS).
* **The Kepler Revolution (2009–2017)**: Stared continuously at ~150,000 stars in Cygnus/Lyra, uncovering that small planets (Super-Earths, Sub-Neptunes) are the most abundant planets in the Galaxy.
* **The TESS & Gaia Era (2018–Present)**: All-sky survey finding bright, nearby transiting planets amenable to atmospheric spectroscopy with JWST, alongside astrometric detections from Gaia.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# 1. Annual Discoveries by Method (Stacked Bar)
ax = axes[0, 0]
method_group = df['discoverymethod'].apply(lambda x: x if x in ['Transit', 'Radial Velocity', 'Microlensing', 'Imaging'] else 'Others')
annual_methods = pd.crosstab(df['disc_year'], method_group)
annual_methods.plot(kind='bar', stacked=True, ax=ax, colormap='tab10', edgecolor='black', linewidth=0.5, alpha=0.85)
ax.set_title('A. Annual Discoveries by Detection Method (1992–2026)', fontsize=12, fontweight='bold')
ax.set_xlabel('Discovery Year', fontsize=11)
ax.set_ylabel('Number of Discoveries', fontsize=11)
ax.tick_params(axis='x', rotation=60, labelsize=8)
ax.legend(title='Method', frameon=True)

# 2. Cumulative Discoveries Highlighting Key Missions
ax = axes[0, 1]
cum_discoveries = df.groupby('disc_year').size().cumsum()
ax.plot(cum_discoveries.index, cum_discoveries.values, color='#08519c', linewidth=2.5, marker='o', markersize=4)

# Annotations for missions (guard against missing years)
for year, label, xytext in [(2009, 'Kepler Launch (2009)', (2000, 1500)),
                              (2016, 'Kepler 2014 & 2016\nBatch Releases', (2006, 3800)),
                              (2018, 'TESS Launch (2018)', (2012, 5500))]:
    if year in cum_discoveries.index:
        ax.annotate(label, xy=(year, cum_discoveries.loc[year]), xytext=xytext,
                    arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=6), fontweight='bold')

ax.set_title('B. Cumulative Exoplanet Discoveries Over Time', fontsize=12, fontweight='bold')
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Cumulative Confirmed Planets', fontsize=11)

# 3. Evolution of Planet Types Over Time
ax = axes[1, 0]
annual_types = pd.crosstab(df['disc_year'], df['planet_type'], normalize='index') * 100
plot_types = [c for c in ['Hot Jupiter', 'Gas Giant', 'Sub-Neptune', 'Super-Earth', 'Terrestrial'] if c in annual_types.columns]
annual_types[plot_types].plot(kind='area', stacked=True, ax=ax, colormap='viridis', alpha=0.85)
ax.set_title('C. Evolution of Detected Planet Types (Relative %)', fontsize=12, fontweight='bold')
ax.set_xlabel('Discovery Year', fontsize=11)
ax.set_ylabel('Proportion of Annual Discoveries (%)', fontsize=11)
ax.legend(loc='upper left', frameon=True, fontsize=9)

# 4. Evolution of Discovery Distance (pc)
ax = axes[1, 1]
sns.boxplot(data=df[df['disc_year'] >= 2000], x='disc_year', y='sy_dist', ax=ax, color='#41b6c4', fliersize=1)
ax.set_yscale('log')
ax.set_title('D. Distance of Discovered Planetary Systems Over Time', fontsize=12, fontweight='bold')
ax.set_xlabel('Discovery Year', fontsize=11)
ax.set_ylabel('System Distance [pc] (log scale)', fontsize=11)
ax.tick_params(axis='x', rotation=60, labelsize=8)

plt.suptitle('Section 4 – Discovery Trends Over Time', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('figures/04_discovery_trends.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure saved: figures/04_discovery_trends.png")

# Print annual discovery statistics
print("\nTop 10 most productive discovery years:")
print(df.groupby('disc_year').size().sort_values(ascending=False).head(10).to_string())


**What this shows**: Exoplanet discoveries grew exponentially during the Kepler era (2009–2017), with batch confirmations producing step-function jumps in the cumulative curve. The composition of confirmed planets shifted from Hot Jupiters (pre-2009) to Sub-Neptunes and Super-Earths (post-2009).

**What it does not establish**: This temporal shift reflects changes in *survey capabilities*, not evolution of planetary populations. Pre-Kepler surveys were not sensitive to small planets, so their absence in the pre-2009 sample is an instrument artifact.


---
# Section 5: Missing-Data Analysis

### Systematic Missingness Patterns
In astrophysical databases, data is rarely "missing at random" (MAR). The structure of missingness encodes physical and observational constraints:

1. **Detection Method Constraints**:
   * **Transit planets**: Directly yield radius $R_p$ from transit depth, but require RV Doppler follow-up or Transit Timing Variations (TTVs) to obtain mass. Many transit hosts are too faint for precision Doppler spectroscopy, leaving $M_p$ unmeasured.
   * **Radial Velocity planets**: Yield $M_p \sin i$ directly but no radius information unless the planet also transits.

2. **Target Brightness**: Faint host stars in the distant Kepler field ($V > 14$) are too dim for precision ground-based Doppler spectroscopy, leaving thousands of transiting sub-Neptunes with unmeasured masses.

3. **Survey Epochs**: Host star distances and luminosities prior to Gaia DR2 (2018) had significant uncertainties and missingness.

> **Caution on MNAR interpretation**: The co-missingness patterns shown below are *consistent with* Missing Not At Random (MNAR) mechanisms driven by physical detection thresholds, but cannot be formally proven to be MNAR from correlation analysis alone. MNAR requires modeling the missingness mechanism itself, which is beyond the scope of this EDA.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# 1. Missingness by Feature
ax = axes[0, 0]
core_cols = ['pl_bmasse', 'pl_rade', 'pl_dens', 'pl_orbper', 'pl_orbsmax', 'pl_orbeccen',
             'pl_insol', 'pl_eqt', 'st_teff', 'st_rad', 'st_mass', 'st_met', 'st_logg', 'st_lum', 'sy_dist', 'sy_vmag']
missing_rates = (df[core_cols].isnull().mean() * 100).sort_values(ascending=False)
bars = ax.barh(missing_rates.index, missing_rates.values, color='#74a9cf', edgecolor='black', alpha=0.85)
ax.set_title('A. Missing Data Percentage Across Key Parameters', fontsize=12, fontweight='bold')
ax.set_xlabel('Missing Rate (%)', fontsize=11)
ax.set_ylabel('Parameter', fontsize=11)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.3, bar.get_y() + bar.get_height()/2.0, f'{w:.1f}%', va='center', fontsize=9)

# 2. Missing Mass and Radius Across Top Detection Methods
# CLARIFICATION:
#   - For Transit planets: we report fraction WITHOUT empirical dynamical mass
#     (is_measured_mass == False means mass is model-derived, not measured directly by Doppler/TTV)
#   - For RV planets: we report fraction with MISSING radius (pl_rade is NaN),
#     because RV yields no geometric transit information
ax = axes[0, 1]
method_miss = []
for m in ['Transit', 'Radial Velocity', 'Microlensing', 'Imaging']:
    sub_m = df[df['discoverymethod'] == m]
    n = len(sub_m)
    # Transit: fraction without empirical (measured) dynamical mass
    no_empirical_mass_pct = (~sub_m['is_measured_mass']).mean() * 100
    # All: fraction with missing radius
    no_radius_pct = sub_m['pl_rade'].isna().mean() * 100
    method_miss.append({
        'Method': m,
        'N': n,
        'No Empirical Mass (%)': round(no_empirical_mass_pct, 1),
        'Missing Radius (%)': round(no_radius_pct, 1)
    })
miss_df = pd.DataFrame(method_miss).set_index('Method')
print("Missing-data rates by method:")
print(miss_df.to_string())

miss_df[['No Empirical Mass (%)', 'Missing Radius (%)']].plot(
    kind='bar', ax=ax, color=['#e41a1c', '#377eb8'], edgecolor='black', alpha=0.85
)
ax.set_title('B. No Empirical Mass vs. Missing Radius by Method\n'
             '(Transit: fraction w/o measured mass; All: fraction w/o measured radius)',
             fontsize=11, fontweight='bold')
ax.set_ylabel('Rate (%)', fontsize=11)
ax.tick_params(axis='x', rotation=15)
ax.legend(frameon=True)

# 3. Missingness by Discovery Era
ax = axes[1, 0]
df['era'] = pd.cut(df['disc_year'], bins=[1990, 2008, 2017, 2027],
                   labels=['Pre-Kepler (≤2008)', 'Kepler Era (2009–2017)', 'TESS/Modern (2018–2026)'])
era_miss = df.groupby('era', observed=True)[['st_met', 'pl_orbeccen', 'sy_dist', 'st_teff']].apply(
    lambda g: g.isna().mean() * 100
)
era_miss.T.plot(kind='bar', ax=ax, colormap='Accent', edgecolor='black', alpha=0.85)
ax.set_title('C. Parameter Missingness Across Space Discovery Eras', fontsize=12, fontweight='bold')
ax.set_xlabel('Parameter', fontsize=11)
ax.set_ylabel('Missing Rate (%)', fontsize=11)
ax.tick_params(axis='x', rotation=20)
ax.legend(title='Era', frameon=True)

# 4. Missingness Correlation Heatmap
ax = axes[1, 1]
miss_matrix = df[core_cols].isnull().astype(int)
miss_matrix = miss_matrix.loc[:, miss_matrix.sum() > 0]
miss_corr = miss_matrix.corr()
sns.heatmap(miss_corr, cmap='vlag', vmin=-1, vmax=1, ax=ax, cbar_kws={'label': 'Pearson Correlation'})
ax.set_title('D. Missingness Correlation Matrix\n(Co-missing Parameter Pairs)', fontsize=12, fontweight='bold')
ax.tick_params(axis='x', rotation=45, labelsize=9)
ax.tick_params(axis='y', labelsize=9)

plt.suptitle('Section 5 – Missing Data Analysis', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('figures/05_missing_data_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
print("\nFigure saved: figures/05_missing_data_analysis.png")

# Specific missing-data statistics for the text
transit_sub = df[df['discoverymethod'] == 'Transit']
rv_sub = df[df['discoverymethod'] == 'Radial Velocity']
transit_no_mass_frac = (~transit_sub['is_measured_mass']).mean()
rv_no_radius_frac = rv_sub['pl_rade'].isna().mean()
print(f"\nTransit planets without empirical dynamical mass: {transit_no_mass_frac:.1%} "
      f"(is_measured_mass == False, N={len(transit_sub):,})")
print(f"RV planets with missing radius (pl_rade is NaN): {rv_no_radius_frac:.1%} "
      f"(no geometric transit, N={len(rv_sub):,})")


**What this shows**: Missingness is highly structured by detection method and observation era. Transit planets frequently lack empirically measured dynamical masses because precision Doppler spectroscopy requires bright host stars. RV planets lack radii because they were not observed to transit. This is consistent with MNAR mechanisms driven by physical detection thresholds.

**Important distinction**: For Transit planets, "No Empirical Mass" means `is_measured_mass == False` — the mass entry is a model-derived estimate (e.g., from mass-radius relations), not a direct Doppler or TTV measurement. For RV planets, "Missing Radius" means `pl_rade` is `NaN` — no photometric transit has been detected.

**What it does not establish**: These co-missingness patterns are *consistent with* Missing Not At Random (MNAR) mechanisms driven by physical detection thresholds, but cannot be formally proven to be MNAR from correlation analysis alone. Formal MNAR testing requires specifying and fitting the missingness mechanism as a latent model.


## Summary of Findings (Sections 2–5)

1. **Exoplanet Demographics**: The confirmed exoplanet population is dominated by Sub-Neptunes ($2–4\,R_\oplus$) and Super-Earths ($1.25–2\,R_\oplus$), planet classes completely absent from our Solar System. The bimodal radius distribution demonstrates the **Fulton Radius Valley** (~1.5–2.0 $R_\oplus$), consistent with photoevaporation and core-powered mass loss models — though both mechanisms are observationally degenerate here.

2. **Host-Star Relationships**: Stars hosting giant planets ($M_p > 100\,M_\oplus$) exhibit higher median metallicity than those hosting small rocky planets, validating the trend predicted by Core Accretion theory. Spearman correlations confirm the mass-metallicity relationship is monotonic and robust.

3. **Detection Method Biases**: Kolmogorov-Smirnov and Mann-Whitney U tests confirm that Transit and Radial Velocity populations differ significantly across all tested parameters ($D = 0.49–0.58$, $p < 10^{-200}$). Transit planets have significantly smaller radii (median ~2.5 $R_\oplus$) and shorter orbital periods (median ~8 days); RV planets have larger masses (median ~350 $M_\oplus$) and longer periods (median ~305 days). These differences reflect instrument sensitivity functions, not the intrinsic galactic planet distribution.

4. **Historical Shifts**: Discoveries transitioned from close-in Hot Jupiters (1990s–2000s) to intermediate sub-Neptunes during Kepler (2009–2017) and nearby bright stars during TESS (2018–present). This reflects instrument capability evolution, not a change in which planets exist.

5. **Systematic Missingness**: Missingness is structured by physics and instrumentation. Transit planets frequently lack empirical dynamical masses; RV planets lack measured radii. These patterns are consistent with MNAR mechanisms but cannot be formally proven MNAR without modeling the missingness mechanism explicitly.

> **Overall caveat**: All findings in this notebook describe the *observed* exoplanet catalog, not an unbiased survey of galactic planetary demographics. Occurrence rate estimates require survey completeness corrections beyond the scope of this EDA.
